# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Source: `docs/flyrank-seo-research-march-2026.pdf`, the FlyRank "State of AI-Driven SEO" report.

**Finding 1 -- ML Appendix, "What Predicts Health?" (p.27).** A Random Forest, holdout-tested, ranks feature importance for predicting `health_score`: Average Position 43%, Impressions 32%, Scroll Depth 15% (90% of total importance in three features). **Where the label comes from:** the paper's own Health Score formula (p.5) is `Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)` -- a weighted SUM of exactly the top three "predictors" the model found, plus CTR. **Does the validation carry the claim?** No -- and a holdout split can't fix this. This is the leakage taxonomy's first pattern: the label is computed FROM the features. An 80/20 split protects against overfitting noise, not against the structural fact that 3 of 4 "predictors" are literally 90% of the formula that produced the target. The paper's own caveat ("importance is descriptive rather than causal") undersells this -- it isn't a causality nuance, the model is reconstructing its own label's recipe. Constructive fix: either drop the label's direct components from the feature set and report what (if anything) predicts health from independent signals, or rename the section plainly as "what health score is made of" rather than "what predicts health."

**Finding 2 -- ML Appendix, "What Predicts Growth?" (p.29).** Logistic Regression, 71% holdout accuracy, separating growing vs. declining pages; "recent impressions" and "days visible" are named among the strongest positive signals. **Where the label comes from:** `trend_direction` (p.5) is "calculated from 30d-vs-prev-30d impression change" -- structurally the same construction as this repo's own `is_declining_label`, which ML-04's contract required excluding `impressions_last_30d`/`impressions_prev_30d` from the feature set because they are `trend_pct`'s literal inputs (proven in ML-07 by recomputing `trend_pct` from them and getting a 100% match). **Does the validation carry the claim?** Unclear, and that's the methodology question: the appendix lists "Impressions" as a feature but doesn't say whether that means the full 90-day total (safe) or also includes the last-30d/prev-30d sub-windows the label is built from (leak-adjacent). Given Finding 1 shows this same paper's ML appendix already has one undisclosed near-tautological setup, and 71% is well within the range a partial leak could produce without being as obvious as Finding 1's 90%-of-formula case, this is worth asking the authors to confirm explicitly, not assuming either way.

In [1]:
# No query needed here -- Section 1 audits the PDF report's own published tables and
# methodology page (docs/flyrank-seo-research-march-2026.pdf, p.5, p.27, p.29), not this repo's data.
print("Health Score formula (p.5): impressions(30) + position(30) + ctr(20) + scroll_depth(20) = 100 pts")
print("Finding 1 top-3 RF importances (p.27): avg_position 43 + impressions 32 + scroll_depth 15 =",
      43 + 32 + 15, "% of total importance -- 3 of the 4 label components")


Health Score formula (p.5): impressions(30) + position(30) + ctr(20) + scroll_depth(20) = 100 pts
Finding 1 top-3 RF importances (p.27): avg_position 43 + impressions 32 + scroll_depth 15 = 90 % of total importance -- 3 of the 4 label components


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

ML-08 already used a grouped client-holdout split. "Before" here means the split I'd get if I hadn't grouped it: a random 75/25 row split, same seed, same model (Logistic Regression), same features. "After" is the ML-08 grouped split, recomputed identically for a clean side-by-side. The honest question a grouped split answers that a random one can't: does this work on a client the model never saw a single page from?

In [2]:
import json
import sys
sys.path.insert(0, "scripts")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = list(MODEL_NUMERIC_FEATURES) + ["has_keyword_data", "has_word_count", "has_scroll_data"]
for col in numeric_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_features = list(MODEL_CATEGORICAL_FEATURES)
for col in categorical_features:
    df[col] = df[col].fillna("unknown").astype(str)

X = pd.concat([df[numeric_features], pd.get_dummies(df[categorical_features], prefix=categorical_features)], axis=1)
y = df["is_declining_label"]

def fit_eval(train_idx, test_idx):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    scaler = StandardScaler()
    X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
    X_test[numeric_features] = scaler.transform(X_test[numeric_features])
    model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    return {
        "n_test": len(y_test),
        "base_rate": round(float(y_test.mean()), 3),
        "roc_auc": round(roc_auc_score(y_test, p), 3),
        "avg_precision": round(average_precision_score(y_test, p), 3),
        "precision_at_20": round(precision_at_k(y_test, p, 20), 3),
        "precision_at_50": round(precision_at_k(y_test, p, 50), 3),
    }

# BEFORE: random row-level split (dishonest -- a client's pages can land on both sides)
rand_train_idx, rand_test_idx = train_test_split(np.arange(len(df)), test_size=0.25, random_state=42, stratify=y)
random_split_clients_shared = len(set(df.iloc[rand_train_idx]["client_id"]) & set(df.iloc[rand_test_idx]["client_id"]))
random_result = fit_eval(rand_train_idx, rand_test_idx)

# AFTER: grouped client-holdout (same split ML-08 used)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
grp_train_idx, grp_test_idx = next(gss.split(df, groups=df["client_id"]))
grouped_result = fit_eval(grp_train_idx, grp_test_idx)

print(f"clients appearing in BOTH train and test under the random split: {random_split_clients_shared} / {df['client_id'].nunique()}")
print("\nBEFORE -- random row-level split:", random_result)
print("AFTER  -- grouped client-holdout: ", grouped_result)
print("\ngap (random minus grouped):", {
    k: round(random_result[k] - grouped_result[k], 3)
    for k in ("roc_auc", "avg_precision", "precision_at_20", "precision_at_50")
})


clients appearing in BOTH train and test under the random split: 31 / 32

BEFORE -- random row-level split: {'n_test': 7500, 'base_rate': 0.542, 'roc_auc': 0.709, 'avg_precision': 0.724, 'precision_at_20': 0.9, 'precision_at_50': 0.86}
AFTER  -- grouped client-holdout:  {'n_test': 7115, 'base_rate': 0.517, 'roc_auc': 0.61, 'avg_precision': 0.605, 'precision_at_20': 0.8, 'precision_at_50': 0.7}

gap (random minus grouped): {'roc_auc': 0.099, 'avg_precision': 0.119, 'precision_at_20': 0.1, 'precision_at_50': 0.16}


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Per the skill's own verify step: deliberately ADD a known-leaky column and confirm the score jumps toward 1.0 -- if it doesn't, the test harness is broken, not the model. `trend_pct` is the ML-04-excluded column (the literal input to the label); adding it here is the attack. Then confirm it's absent from the honest ML-08/Section-2 feature set.

In [3]:
# honest feature set == exactly Section 2 / ML-08's columns, confirmed absent of label-source fields
label_source_cols = {"trend_direction", "trend_pct"}
print("label-source columns present in the honest feature set (should be empty):",
      sorted(label_source_cols & set(X.columns)))

# the attack: add trend_pct back in and refit on the SAME grouped split
df["trend_pct"] = df["trend_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
X_attack = X.copy()
X_attack["trend_pct"] = df["trend_pct"]
attack_numeric_features = numeric_features + ["trend_pct"]

X_train, X_test = X_attack.iloc[grp_train_idx].copy(), X_attack.iloc[grp_test_idx].copy()
y_train, y_test = y.iloc[grp_train_idx], y.iloc[grp_test_idx]
scaler = StandardScaler()
X_train[attack_numeric_features] = scaler.fit_transform(X_train[attack_numeric_features])
X_test[attack_numeric_features] = scaler.transform(X_test[attack_numeric_features])

attack_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
attack_model.fit(X_train, y_train)
p_attack = attack_model.predict_proba(X_test)[:, 1]

attack_result = {
    "roc_auc": round(roc_auc_score(y_test, p_attack), 3),
    "avg_precision": round(average_precision_score(y_test, p_attack), 3),
    "precision_at_20": round(precision_at_k(y_test, p_attack, 20), 3),
    "precision_at_50": round(precision_at_k(y_test, p_attack, 50), 3),
}
print("\nHONEST (Section 2 'after', no trend_pct):", grouped_result)
print("ATTACK (trend_pct added back in):        ", attack_result)

trend_pct_coef = pd.Series(attack_model.coef_[0], index=X_attack.columns)["trend_pct"]
print("\ntrend_pct coefficient (scaled):", round(trend_pct_coef, 2),
      "-- huge and negative: lower trend_pct strongly predicts is_declining_label, exactly as designed, confirming the harness detects a real leak")


label-source columns present in the honest feature set (should be empty): []



HONEST (Section 2 'after', no trend_pct): {'n_test': 7115, 'base_rate': 0.517, 'roc_auc': 0.61, 'avg_precision': 0.605, 'precision_at_20': 0.8, 'precision_at_50': 0.7}
ATTACK (trend_pct added back in):         {'roc_auc': 0.999, 'avg_precision': 0.999, 'precision_at_20': 1.0, 'precision_at_50': 1.0}

trend_pct coefficient (scaled): -48.31 -- huge and negative: lower trend_pct strongly predicts is_declining_label, exactly as designed, confirming the harness detects a real leak


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (from ML-08's own summary):** "Logistic Regression wins outright: Precision@20 0.80 / Precision@50 0.70 vs tie-aware baseline 0.589/0.545 and random forest 0.55/0.56."

That sentence states real numbers but reads as more settled than Section 2 and 3 just showed it is: "wins outright" implies a fixed fact rather than a number measured under one particular honest split, and it doesn't carry the memorization-gap context (Section 2) or the base rate next to it.

**Rewrite, on the claim ladder (validated model, ranking, out-of-sample -- one rung below causal):** "On a client-holdout split -- 8 clients the model never saw a single page from, seed 42 -- Logistic Regression **ranks/flags** declining pages at Precision@50 of 0.70, versus a base rate of 0.517 and a frozen rule baseline of 0.545. The baseline value is tie-aware: only 3 held-out pages receive a positive rule score, so the remaining top-50 slots use the zero-score group's expected hit rate rather than arbitrary row order. On this same feature set scored with a random (non-grouped) split instead, Precision@50 reads 0.86 -- 16 points higher purely from letting a client's other pages leak into training, which is why the grouped number is the one this repo reports as the honest measurement. This is a **decision-support** ranking on one 30k-page snapshot, not a claim that the model finds every declining page, or that its ranking would hold on a different portfolio or time period."

What changed: the model's action is named as a verb the claim ladder allows ("ranks/flags", not "predicts" or "knows"), the base rate sits next to the metric, the split is named explicitly enough to audit, and the random-split number is disclosed alongside the honest one instead of only reporting the flattering split.

In [4]:
# Confirms the rewrite's numbers trace back to what Sections 2-3 actually computed --
# a claim rewrite is only honest if it cites numbers that exist in this notebook's own output.
print("base rate (grouped test split):", grouped_result["base_rate"])
print("Logistic Regression, grouped holdout, precision@50:", grouped_result["precision_at_50"])
with open("work/outputs/baseline_metrics.json") as baseline_file:
    baseline_precision_at_50 = json.load(baseline_file)["precision_at_50_client_holdout"]
print("baseline_rules, grouped holdout, tie-aware precision@50 (from ML-07):", baseline_precision_at_50)
print("Logistic Regression, RANDOM split, precision@50:", random_result["precision_at_50"])
print("gap disclosed in the rewrite:", round(random_result["precision_at_50"] - grouped_result["precision_at_50"], 3))


base rate (grouped test split): 0.517
Logistic Regression, grouped holdout, precision@50: 0.7
baseline_rules, grouped holdout, tie-aware precision@50 (from ML-07): 0.545
Logistic Regression, RANDOM split, precision@50: 0.86
gap disclosed in the rewrite: 0.16


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.